> 在线讲义：[第 35 章 · 卷积的数学：稀疏共享的线性映射、尺寸公式、感受野、卷积定理与 im2col](https://wikieden.github.io/math-and-eval-stats/lessons/35-convolution.html) · 本书《数学与实验统计》

# 第 35 章 · 卷积的数学：稀疏共享的线性映射、尺寸公式、感受野、卷积定理与 im2col

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/35-convolution.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 Toeplitz | y = Tx | `toeplitz_matrix(6, K) @ X` | [−1, 0, −1, −8] |
| 第二步 互相关 | 翻不翻核 | `np.correlate` vs `np.convolve` | 差负号 |
| 第三步 尺寸 | ⌊…⌋ + 1 | `out_size(224, 7, 2, 3)` | 112 |
| 第四步 感受野 | 递推 | `receptive_field(layers)` | 7、15、15 |
| 第五步 im2col | 矩阵形状 | `im2col_cost(64, 56, 56, 3)` | (576, 3136)，9 倍 |

例子：一维信号 [3, 1, 4, 1, 5, 9] 和差分核 [1, 0, −1]（看相邻变化）；尺寸公式、感受野、im2col 的内存账。
里程碑 ME-M23 的“二维 conv2d = im2col × 矩阵乘法并和 PyTorch 对数”“用 FFT 验证卷积定理”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
X = np.array([3.0, 1.0, 4.0, 1.0, 5.0, 9.0])
K = np.array([1.0, 0.0, -1.0])

**函数 `toeplitz_matrix`**：valid 互相关写成矩阵：第 i 行在第 i..i+len(k)−1 列放核，其余是 0。

In [ ]:
def toeplitz_matrix(n: int, k: np.ndarray) -> np.ndarray:
    """valid 互相关写成矩阵：第 i 行在第 i..i+len(k)−1 列放核，其余是 0。"""
    m = n - len(k) + 1
    t = np.zeros((m, n))
    for i in range(m):
        t[i, i:i + len(k)] = k
    return t

**函数 `out_size`**：输出尺寸 ⌊(n + 2p − d(k − 1) − 1)/s⌋ + 1（PyTorch Conv2d 文档的公式）。

In [ ]:
def out_size(n: int, k: int, s: int = 1, p: int = 0, d: int = 1) -> int:
    """输出尺寸 ⌊(n + 2p − d(k − 1) − 1)/s⌋ + 1（PyTorch Conv2d 文档的公式）。"""
    return (n + 2 * p - d * (k - 1) - 1) // s + 1

**函数 `receptive_field`**：layers 是 (k, s, d) 的列表。r ← r + (k − 1)·d·(前面所有层步长的乘积)。

In [ ]:
def receptive_field(layers: list[tuple[int, int, int]]) -> int:
    """layers 是 (k, s, d) 的列表。r ← r + (k − 1)·d·(前面所有层步长的乘积)。"""
    r, jump = 1, 1
    for k, s, d in layers:
        r += (k - 1) * d * jump
        jump *= s
    return r

**函数 `im2col_cost`**：im2col 矩阵的形状 (C·k·k, H_out·W_out)、它的元素数、原输入的元素数。

In [ ]:
def im2col_cost(c: int, h: int, w: int, k: int, s: int = 1, p: int = 1) -> tuple[tuple[int, int], int, int]:
    """im2col 矩阵的形状 (C·k·k, H_out·W_out)、它的元素数、原输入的元素数。"""
    ho, wo = out_size(h, k, s, p), out_size(w, k, s, p)
    shape = (c * k * k, ho * wo)
    return shape, shape[0] * shape[1], c * h * w

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 卷积是一个稀疏、共享权重的线性映射")
t = toeplitz_matrix(len(X), K)
print(f"  Toeplitz 矩阵（4×6）：\n{t.astype(int)}")
print(f"  矩阵乘 T·x = {t @ X}；np.correlate(x, k, 'valid') = {np.correlate(X, K, 'valid')}")
print(f"  参数：全连接 4×6 = 24 个，卷积 3 个（共享）；非零元 {int((t != 0).sum())} / 24")

In [ ]:
print("\n第二步 深度学习里的“卷积”其实是互相关")
print(f"  互相关 {np.correlate(X, K, 'valid')}；数学卷积（核翻转）{np.convolve(X, K, 'valid')}：差一个负号")

In [ ]:
print("\n第三步 尺寸公式")
for args in ((32, 3, 1, 1, 1), (32, 3, 2, 1, 1), (32, 3, 1, 0, 2), (224, 7, 2, 3, 1)):
    print(f"  n = {args[0]}、k = {args[1]}、步长 {args[2]}、填充 {args[3]}、空洞 {args[4]}：输出 {out_size(*args)}")

In [ ]:
print("\n第四步 感受野")
for name, layers in (("三层 3×3", [(3, 1, 1)] * 3), ("三层 3×3、前两层步长 2", [(3, 2, 1), (3, 2, 1), (3, 1, 1)]),
                     ("空洞 1、2、4", [(3, 1, 1), (3, 1, 2), (3, 1, 4)])):
    print(f"  {name}：感受野 {receptive_field(layers)}")

In [ ]:
print("\n第五步 卷积定理与 im2col 的代价")
for n, k in ((1024, 3), (1024, 101), (65536, 1025)):
    print(f"  n = {n}、核长 {k}：直接算 {n * k:,} 次乘法；FFT 约 3·n·log₂n = {int(3 * n * np.log2(n)):,}")
shape, cols, inp = im2col_cost(64, 56, 56, 3)
print(f"  im2col：C = 64、56×56、3×3 核 → 矩阵 {shape}，{cols:,} 个元素，是输入（{inp:,}）的 {cols / inp:.0f} 倍")

## 3. 代码里才看得到的

1. **`np.correlate` 和 `np.convolve` 是两个函数**：NumPy 把互相关和卷积分开命名；PyTorch 的 `conv1d` 对应的是前者。
2. **感受野递推里 jump 的更新在加之后**：这一层的核按"前面各层"的累计步长伸展，自己的步长只影响后面的层。
3. **FFT 的代价只是估算**：3n log₂n 粗略算了两次正变换、一次反变换；实际还要补零到 2 的幂、处理边界。

## 4. 自己改一改

1. 把核换成 [1, 2, 1]/4（平滑），看 Toeplitz 矩阵和输出。
2. 算一算 ResNet-50 第一层（7×7、步长 2）再接 3×3 最大池化（步长 2）以后的感受野。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 卷积是盖印章，矩阵稀疏又共享；
> 框架里不翻核，其实是互相关；
> 输出等于 n 加 2p 减 d(k−1) 减 1，除 s 加 1；
> 感受野逐层加 (k−1)·d·累计步长；
> 小核 im2col 变矩阵乘，长核 FFT 变逐点乘。